<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/EXPERIMENT_10_LangChain_RAG_Chatbot_with_Conversational_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAKESH M A - ENG23CS0168

LangChain RAG Chatbot with Conversational Memory

In [1]:
!pip install torch transformers accelerate sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 96.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_1294/3092554125.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32
).to(DEV)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
@torch.no_grad()
def generate(prompt, max_new_tokens=150):
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True
    )
    enc = tok(text, return_tensors="pt").to(DEV)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)



In [5]:
DOCS = [
    "Dayananda Sagar University offers the course Large Language Models, "
    "course code 23CS4701.",
    "LoRA (Low-Rank Adaptation) freezes pretrained weights and trains small "
    "low-rank matrices, cutting trainable parameters by more than 99 percent.",
    "Retrieval-Augmented Generation splits documents into chunks, embeds them, "
    "stores them in a vector database such as FAISS or Chroma, and inserts the "
    "top-k chunks into the prompt.",
    "Temperature rescales logits before sampling. Top-k keeps the k most likely "
    "tokens; top-p keeps the smallest set of tokens whose cumulative "
    "probability exceeds p.",
]

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
texts = [c for d in DOCS for c in splitter.split_text(d)]
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
retriever = FAISS.from_texts(texts, emb).as_retriever(search_kwargs={"k": 3})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful knowledge assistant. Use the context to answer; "
               "if it is not in the context, say you don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

In [8]:
llm = RunnableLambda(lambda pv: generate(pv.to_string()))

In [9]:
chain = (
    RunnablePassthrough.assign(
        context=lambda x: "\n".join(
            d.page_content for d in retriever.invoke(x["question"])
        )
    )
    | prompt
    | llm
)

In [10]:
store = {}
def get_history(session_id):
    return store.setdefault(session_id, InMemoryChatMessageHistory())

app = RunnableWithMessageHistory(
    chain, get_history,
    input_messages_key="question", history_messages_key="history"
)
cfg = {"configurable": {"session_id": "demo"}}

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:


for q in ["What is LoRA?",
          "How many parameters does it save?",
          "What is the course code of the LLM course?",
          "Who is the president of Mars?"]:
    print(f"\nUser: {q}\nBot : {app.invoke({'question': q}, config=cfg)}")

/tmp/ipykernel_1294/601422016.py:3: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  return store.setdefault(session_id, InMemoryChatMessageHistory())



User: What is LoRA?
Bot : LoRA stands for "Low-Rank Adaptation." It's a technique used in natural language processing (NLP) that aims to improve the performance of models trained on large datasets by reducing their size while maintaining good performance. Specifically, LoRA involves:

1. **Freezing Pretrained Weights**: This means that the model's weights are frozen during training, meaning they remain constant throughout the process without being updated. Instead, these weights are set to zero after each epoch.

2. **Training Small Low-Rank Matrices**: The model is then trained using smaller, lower-dimensional representations called low-rank matrices. These matrices are typically sparse and have fewer non-zero elements compared to the full matrix. By doing this, the model can learn more

User: How many parameters does it save?
Bot : LoRA saves around 99% of the parameters in a pre-trained model. This means that even though the model has been trained with a lot of data, it retains onl

In [12]:
print("\n--- interactive mode (type 'exit' to quit) ---")
while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit", ""}:
        break
    print("Bot:", app.invoke({"question": q}, config=cfg))


--- interactive mode (type 'exit' to quit) ---
You: HI
Bot: Hello! Is there anything specific you would like to ask or discuss? I'm here to assist you with any questions you might have.
You: HOW ARE YOU
Bot: I am functioning as a virtual AI assistant designed to provide information and assistance based on the context provided. I do not have personal feelings or emotions, nor do I engage in conversation about myself. My purpose is to offer useful responses to your inquiries. How can I assist you today?
You: EXIT
